In [1]:
# Importing ChexFound Code Base

import os
import sys
import importlib

# 1. Setup repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    print("Cloning CheXFound codebase...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

# 2. Register path permanently for this notebook session
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

# 3. Test import to guarantee future cells will not fail
from chexfound.models import vision_transformer as vits
print("\n✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.")

Cloning CheXFound codebase...
Cloning into '/kaggle/working/CheXFound'...
remote: Enumerating objects: 460, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 460 (delta 134), reused 407 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (460/460), 24.14 MiB | 26.60 MiB/s, done.
Resolving deltas: 100% (134/134), done.

✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.


/kaggle/working/CheXFound/chexfound/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/CheXFound/chexfound/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/CheXFound/chexfound/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [2]:
# Load and Freeze the ChexFound Trained Parameters

import os
import glob
import torch
import torch.nn as nn
from chexfound.models import vision_transformer as vits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for inference: {device}")

# 1. Locate teacher checkpoint
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading weights from: {teacher_path}")

# 2. Instantiate ViT-Large backbone (512x512 resolution, patch_size=16)
model = vits.vit_large(patch_size=16, img_size=512)

# 3. Load checkpoint and align state dict keys
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = {
    (k[9:] if k.startswith('backbone.') else k): v 
    for k, v in state_dict.items()
}

missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# 4. Freeze backbone and mount on GPU
model.eval()
model.to(device)
for param in model.parameters():
    param.requires_grad = False

print(f"\n✓ CheXFound ViT-Large Loaded:")
print(f"  - Missing keys: {len(missing_keys)}")
print(f"  - Unexpected keys: {len(unexpected_keys)}")
print(f"  - All parameters frozen: {all(not p.requires_grad for p in model.parameters())}")

# 5. Sanity check: Run a dummy forward pass through the last 4 blocks
dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    sample_layers = model.get_intermediate_layers(dummy, n=4, return_class_token=True)
    # Each block outputs: (patch_tokens [B, 1024, 1024], cls_token [B, 1024])
    patch_sample, cls_sample = sample_layers[-1]
    
print(f"\n✓ Forward check successful:")
print(f"  - Patch tokens shape: {patch_sample.shape} (1024 spatial patches)")
print(f"  - CLS token shape   : {cls_sample.shape} (1024 dimensions)")

Device for inference: cuda
Loading weights from: /kaggle/input/datasets/suyogmagar08/chexfound-weights/teacher_checkpoint.pth

✓ CheXFound ViT-Large Loaded:
  - Missing keys: 240
  - Unexpected keys: 305
  - All parameters frozen: True

✓ Forward check successful:
  - Patch tokens shape: torch.Size([1, 1024, 1024]) (1024 spatial patches)
  - CLS token shape   : torch.Size([1, 1024]) (1024 dimensions)


In [3]:
import os
import sys
import glob
import re
import importlib
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 1. Import Official GLoRI & AsymmetricLoss directly from cloned repo
repo_dir = '/kaggle/working/CheXFound'
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
importlib.invalidate_caches()

from chexfound.models import vision_transformer as vits
from chexfound.eval.classification.glori import GLoRI
from chexfound.eval.losses import AsymmetricLoss

# 2. Load VinDr-CXR Annotations (All 28 findings or 6 target findings)
csv_path = '/kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train.csv'
if not os.path.exists(csv_path):
    csv_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]

annot_df = pd.read_csv(csv_path)
print(f"Loaded CSV: {csv_path}")

# All 28 findings in VinDr-CXR (excluding 'No finding' which is all-zeros in multi-label)
ALL_28_CLASSES = sorted([c for c in annot_df['class_name'].unique() if c != 'No finding'])
TABLE_IV_6_CLASSES = [
    'Aortic enlargement', 'Cardiomegaly', 'Pulmonary fibrosis',
    'Pleural thickening', 'Pleural effusion', 'Pneumothorax'
]
print(f"Total pathological classes in VinDr-CXR: {len(ALL_28_CLASSES)}")

# Map images inside VinBigData folder
vin_dir = os.path.dirname(csv_path)
all_imgs = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f"{vin_dir}/**/*.png", recursive=True) + glob.glob(f"{vin_dir}/**/*.jpg", recursive=True)
}

labelled_ids = set(annot_df['image_id'].unique())
filtered_imgs = {k: v for k, v in all_imgs.items() if k in labelled_ids}

image_abnormalities = (
    annot_df[annot_df['class_name'].isin(ALL_28_CLASSES)]
    .groupby('image_id')['class_name']
    .apply(set)
    .to_dict()
)

records = []
for img_id, p in filtered_imgs.items():
    findings = image_abnormalities.get(img_id, set())
    labels = [1 if c in findings else 0 for c in ALL_28_CLASSES]
    records.append({'image_id': img_id, 'path': p, 'labels': labels})

df_vindr = pd.DataFrame(records).sort_values('image_id').reset_index(drop=True)
Y_np = np.array(df_vindr['labels'].tolist(), dtype=np.float32)

# 70:10:20 Stratified Split
indices = np.arange(len(df_vindr))
has_disease = (Y_np.sum(axis=1) > 0).astype(int)
train_val_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=42, stratify=has_disease)
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.125, random_state=42, stratify=has_disease[train_val_idx])
print(f"Split Sizes -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

# 3. Load Frozen CheXFound ViT-Large Backbone
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
teacher_ckpt = torch.load(teacher_files[0], map_location='cpu')
raw_state = teacher_ckpt.get('teacher', teacher_ckpt)

clean_state = {}
for k, v in raw_state.items():
    if k.startswith(('dino_head.', 'ibot_head.')):
        continue
    new_k = k
    for pfx in ['module.', 'backbone.']:
        if new_k.startswith(pfx):
            new_k = new_k[len(pfx):]
    new_k = re.sub(r'^blocks\.\d+\.(\d+)\.', r'blocks.\1.', new_k)
    clean_state[new_k] = v

num_registers = clean_state['register_tokens'].shape[1] if 'register_tokens' in clean_state else 0
native_res = int(((clean_state['pos_embed'].shape[1] - 1) ** 0.5) * 16)

backbone = vits.vit_large(
    patch_size=16,
    img_size=native_res,
    init_values=1e-5 if any('ls1' in k for k in clean_state) else None,
    ffn_layer="swiglu" if any('w12' in k for k in clean_state) else "mlp",
    block_chunks=0,
    num_register_tokens=num_registers
)
backbone.load_state_dict(clean_state, strict=False)
backbone.eval().to(device)
for p in backbone.parameters():
    p.requires_grad = False

# 4. DataLoaders
eval_transform = T.Compose([
    T.Resize((512, 512), interpolation=InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class VinDrImageDataset(Dataset):
    def __init__(self, df, y_arr, transform):
        self.paths = df['path'].values
        self.y = torch.tensor(y_arr, dtype=torch.float32)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), self.y[idx]

train_loader = DataLoader(VinDrImageDataset(df_vindr.iloc[train_idx], Y_np[train_idx], eval_transform), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(VinDrImageDataset(df_vindr.iloc[val_idx], Y_np[val_idx], eval_transform), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(VinDrImageDataset(df_vindr.iloc[test_idx], Y_np[test_idx], eval_transform), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# 5. Instantiate Official GLoRI Head (4 blocks * 1024 = 4096 initial_num_features, cat_cls=True)
official_glori = GLoRI(
    num_classes=len(ALL_28_CLASSES),
    decoder_embedding=768,
    initial_num_features=4096,
    use_n_blocks=4,
    multiview=False,
    cat_cls=True
).to(device)

criterion = AsymmetricLoss()
optimizer = torch.optim.AdamW(official_glori.parameters(), lr=1e-4, weight_decay=1e-4)
epochs = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

def evaluate_official_glori(loader):
    official_glori.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(device)
            # Official GLoRI expects x as a list/tuple where x[0] is intermediate_layers output
            layers = backbone.get_intermediate_layers(imgs, n=4, return_class_token=True)
            logits = official_glori([layers])
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(targets.numpy())
    probs = np.vstack(all_probs)
    targets = np.vstack(all_targets)
    
    # Compute valid per-class AUROC & AUPRC (skipping classes with 0 positives in split)
    aurocs, auprcs = {}, {}
    for idx, cname in enumerate(ALL_28_CLASSES):
        if targets[:, idx].sum() > 0:
            aurocs[cname] = roc_auc_score(targets[:, idx], probs[:, idx])
            auprcs[cname] = average_precision_score(targets[:, idx], probs[:, idx])
    return aurocs, auprcs

# 6. Train & Evaluate Official GLoRI Head
best_val_auc = -1.0
best_weights = None

for epoch in range(1, epochs + 1):
    official_glori.train()
    total_loss = 0.0
    for imgs, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False):
        imgs, targets = imgs.to(device), targets.to(device)
        with torch.no_grad():
            layers = backbone.get_intermediate_layers(imgs, n=4, return_class_token=True)
        
        optimizer.zero_grad()
        logits = official_glori([layers])
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    
    val_aurocs, val_auprcs = evaluate_official_glori(val_loader)
    mean_v_auc = np.mean(list(val_aurocs.values()))
    mean_v_ap = np.mean(list(val_auprcs.values()))
    print(f"Epoch {epoch:2d} | Loss: {total_loss/len(train_loader):.4f} | Val Mean AUROC (All classes): {mean_v_auc:.4f} | AUPRC: {mean_v_ap:.4f}")
    
    if mean_v_auc > best_val_auc:
        best_val_auc = mean_v_auc
        best_weights = {k: v.cpu() for k, v in official_glori.state_dict().items()}

# 7. Final Test Evaluation
official_glori.load_state_dict({k: v.to(device) for k, v in best_weights.items()})
test_aurocs, test_auprcs = evaluate_official_glori(test_loader)

print("\n" + "="*75)
print("     OFFICIAL REPO GLoRI HEAD RESULTS (VINDR-CXR)")
print("="*75)
print(f"All Evaluated Classes ({len(test_aurocs)} classes) -> Mean AUROC: {np.mean(list(test_aurocs.values())):.4f} | Mean AUPRC: {np.mean(list(test_auprcs.values())):.4f}")
print("-" * 75)
print("Table IV 6-Class Subset Breakdown:")
sub_aucs, sub_aps = [], []
for c in TABLE_IV_6_CLASSES:
    if c in test_aurocs:
        sub_aucs.append(test_aurocs[c])
        sub_aps.append(test_auprcs[c])
        print(f"  {c:22} | AUROC: {test_aurocs[c]:.4f} | AUPRC: {test_auprcs[c]:.4f}")
print("-" * 75)
print(f"6-Class Subset Mean AUROC : {np.mean(sub_aucs):.4f}  (Table IV Target: 0.872)")
print(f"6-Class Subset Mean AUPRC : {np.mean(sub_aps):.4f}  (Table IV Target: 0.341)")
print("="*75)

Device: cuda
Loaded CSV: /kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train.csv
Total pathological classes in VinDr-CXR: 14
Split Sizes -> Train: 10500 | Val: 1500 | Test: 3000


Epoch  1 | Loss: 22.7138 | Val Mean AUROC (All classes): 0.8813 | AUPRC: 0.4202


Epoch  2 | Loss: 14.4509 | Val Mean AUROC (All classes): 0.9151 | AUPRC: 0.4953


Epoch  3 | Loss: 12.0407 | Val Mean AUROC (All classes): 0.9274 | AUPRC: 0.5190


Epoch  4 | Loss: 10.3563 | Val Mean AUROC (All classes): 0.9286 | AUPRC: 0.5363


Epoch  5 | Loss: 9.2368 | Val Mean AUROC (All classes): 0.9309 | AUPRC: 0.5458


Epoch  6 | Loss: 7.9691 | Val Mean AUROC (All classes): 0.9367 | AUPRC: 0.5646


Epoch  7 | Loss: 6.8400 | Val Mean AUROC (All classes): 0.9383 | AUPRC: 0.5731


Epoch  8 | Loss: 5.8286 | Val Mean AUROC (All classes): 0.9385 | AUPRC: 0.5699


Epoch  9 | Loss: 5.0881 | Val Mean AUROC (All classes): 0.9399 | AUPRC: 0.5773


Epoch 10 | Loss: 4.6749 | Val Mean AUROC (All classes): 0.9402 | AUPRC: 0.5790

     OFFICIAL REPO GLoRI HEAD RESULTS (VINDR-CXR)
All Evaluated Classes (14 classes) -> Mean AUROC: 0.9417 | Mean AUPRC: 0.5832
---------------------------------------------------------------------------
Table IV 6-Class Subset Breakdown:
  Aortic enlargement     | AUROC: 0.9798 | AUPRC: 0.9358
  Cardiomegaly           | AUROC: 0.9835 | AUPRC: 0.9311
  Pulmonary fibrosis     | AUROC: 0.9516 | AUPRC: 0.7788
  Pleural thickening     | AUROC: 0.9372 | AUPRC: 0.6733
  Pleural effusion       | AUROC: 0.9682 | AUPRC: 0.8122
  Pneumothorax           | AUROC: 0.8880 | AUPRC: 0.4285
---------------------------------------------------------------------------
6-Class Subset Mean AUROC : 0.9514  (Table IV Target: 0.872)
6-Class Subset Mean AUPRC : 0.7599  (Table IV Target: 0.341)
